# Potential Wrappers

galpy's wrapper potentials modify existing potentials by adding time dependence,
rotation, tilting, or amplitude modulation. This notebook demonstrates the most
commonly used wrappers.

In [ ]:
%matplotlib inline
import numpy
from matplotlib import pyplot as plt
from galpy import potential
from galpy.potential import (
    MWPotential2014,
    DehnenBarPotential,
    DehnenSmoothWrapperPotential,
    SolidBodyRotationWrapperPotential,
    RotateAndTiltWrapperPotential,
    GaussianAmplitudeWrapperPotential,
    TimeDependentAmplitudeWrapperPotential,
    LogarithmicHaloPotential,
    MiyamotoNagaiPotential,
)
from galpy.orbit import Orbit

## Growing a bar with DehnenSmoothWrapperPotential

A common use case is to smoothly grow a bar potential over some time period.
`DehnenSmoothWrapperPotential` multiplies a potential by a smooth function that
transitions from 0 to 1.

In [ ]:
# Create a Dehnen bar potential
bar = DehnenBarPotential(omegab=1.8, rb=0.4, Af=0.02, tform=-2.0, tsteady=0.5)

# Wrap it with a smooth growth function
bar_grown = DehnenSmoothWrapperPotential(pot=bar, tform=-2.0, tsteady=0.5)

# Show the amplitude over time
ts = numpy.linspace(-3.0, 0.0, 201)
amps = [bar_grown(1.0, 0.0, phi=0.0, t=t) for t in ts]
plt.plot(ts, amps)
plt.xlabel(r"$t$")
plt.ylabel(r"$\Phi(R=1, z=0, \phi=0)$")
plt.title("Bar potential growing over time")

## SolidBodyRotationWrapperPotential

This wrapper rotates a potential at a constant angular velocity. This is useful
for adding a rotating bar to an axisymmetric background.

In [ ]:
# A triaxial logarithmic halo potential, made to rotate
lp = LogarithmicHaloPotential(normalize=1.0, b=0.8, q=0.9)
lp_rot = SolidBodyRotationWrapperPotential(pot=lp, omega=1.5)

# Evaluate at different times to see the rotation
phis = numpy.linspace(0, 2 * numpy.pi, 100)
for t in [0.0, 0.5, 1.0]:
    vals = [lp_rot(1.0, 0.0, phi=phi, t=t) for phi in phis]
    plt.plot(phis, vals, label=f"t = {t:.1f}")
plt.xlabel(r"$\phi$")
plt.ylabel(r"$\Phi$")
plt.legend()
plt.title("Rotating potential at different times")

## RotateAndTiltWrapperPotential

This wrapper rotates and tilts a potential with respect to the default
coordinate system. This is useful for, e.g., modeling a tilted dark-matter halo.

In [ ]:
# Tilt a flattened logarithmic potential by 30 degrees
lp_flat = LogarithmicHaloPotential(normalize=1.0, q=0.7)
lp_tilted = RotateAndTiltWrapperPotential(
    pot=lp_flat, zvec=[0.0, numpy.sin(numpy.pi / 6), numpy.cos(numpy.pi / 6)]
)

# Compare densities in the (x, z) plane
potential.plotDensities(
    lp_tilted, rmin=0.1, rmax=2.0, zmin=-1.0, zmax=1.0, nrs=50, nzs=50, log=True
)

## GaussianAmplitudeWrapperPotential

Multiplies a potential by a Gaussian in time, useful for transient perturbations:

In [ ]:
disk = MiyamotoNagaiPotential(a=0.5, b=0.0375, normalize=0.1)
transient = GaussianAmplitudeWrapperPotential(pot=disk, to=0.0, sigma=0.5)

ts = numpy.linspace(-2.0, 2.0, 201)
amps = [transient(1.0, 0.0, t=t) for t in ts]
plt.plot(ts, amps)
plt.xlabel(r"$t$")
plt.ylabel(r"$\Phi$")
plt.title("Gaussian amplitude modulation")

## TimeDependentAmplitudeWrapperPotential

Use an arbitrary function of time to modulate the amplitude:

In [ ]:
# Sinusoidal amplitude modulation
disk = MiyamotoNagaiPotential(a=0.5, b=0.0375, normalize=0.1)
oscillating = TimeDependentAmplitudeWrapperPotential(
    pot=disk, A=lambda t: 1.0 + 0.5 * numpy.sin(4.0 * numpy.pi * t)
)

ts = numpy.linspace(0.0, 2.0, 201)
amps = [oscillating(1.0, 0.0, t=t) for t in ts]
plt.plot(ts, amps)
plt.xlabel(r"$t$")
plt.ylabel(r"$\Phi$")
plt.title("Time-dependent amplitude modulation")

## Adjusting amplitude by multiplying

You can multiply a potential by a number to adjust its amplitude:

In [ ]:
lp = LogarithmicHaloPotential(normalize=1.0)
print("Original:", lp(1.0, 0.0))
lp_half = 0.5 * lp
print("Halved:", lp_half(1.0, 0.0))

## Combining wrappers with background potentials

A typical use case: grow a rotating bar on top of an axisymmetric MW potential
and integrate an orbit:

In [ ]:
# Axisymmetric background
bg = MWPotential2014

# Dehnen bar, smoothly grown
bar = DehnenBarPotential(omegab=1.8, rb=0.4, Af=0.02, tform=-2.0, tsteady=0.5)
bar_smooth = DehnenSmoothWrapperPotential(pot=bar, tform=-2.0, tsteady=0.5)

# Full potential = background + bar
full_pot = bg + bar_smooth

# Integrate an orbit
o = Orbit([1.0, 0.1, 1.1, 0.0, 0.05, 0.0])
ts = numpy.linspace(0.0, 10.0, 10001)
o.integrate(ts, full_pot)
o.plot()